# External validation on CKD-ROUTE

Does the CKD-PC 40% eGFR decline equation rank and calibrate risk correctly in a cohort
it was not developed on?

**Data:** CKD-ROUTE, a prospective cohort of 1,138 newly referred CKD stage G2-G5
patients in Japan, with eGFR measured every 6 months for up to 36 months.
Iimori S, et al. *Prognosis of chronic kidney disease with normal-range proteinuria:
The CKD-ROUTE study.* PLOS ONE 2018. Data on Dryad under CC0:
[doi:10.5061/dryad.kq23s](https://doi.org/10.5061/dryad.kq23s).

Download `ROUTE_proteinuria_dataset.xlsx` from Dryad into `data/raw/`.

## 1. Load the data

Load the spreadsheet and confirm two things about how it is coded: the `eGFR` column
uses the Japanese (JSN) equation, and `gender` 2 means female. The JSN equation is
eGFR = 194 × Cr^-1.094 × age^-0.287 × 0.739 (if female).

In [ ]:
import numpy as np
import pandas as pd

# Raw CKD-ROUTE file from Dryad (data/ is gitignored)
route = pd.read_excel("../data/raw/ROUTE_proteinuria_dataset.xlsx")
print(f"{len(route):,} patients, {route.shape[1]} columns")

# Recompute the JSN eGFR, assuming gender 2 = female
jsn = 194 * route["Cr"] ** -1.094 * route["age"] ** -0.287 * np.where(route["gender"] == 2, 0.739, 1)
print("eGFR column is the JSN equation, gender 2 = female:",
      np.allclose(jsn, route["eGFR"], rtol=1e-3))

route.head()

## 2. Recode to the model's inputs

| Model input | CKD-ROUTE source | Coding |
|---|---|---|
| eGFR | `Cr` (mg/dL) | recomputed with CKD-EPI 2021, since `eGFR` is the JSN equation |
| uACR | `UPCR` (g/gCr) | × 1000 to mg/g, then `pcr_to_acr()` (Sumida crude equation) |
| BP medication | `use of RAASi`, `use of CCB`, `use of diuretics` | 1 if any |
| CHD | `prevalence of CVD` | **assumption:** prior CVD counted as CHD |
| Heart failure, atrial fibrillation | not recorded | **assumption:** 0 |
| Smoking | not recorded | **assumption:** never (the model's reference) |
| HbA1c, diabetes medication | not recorded | **assumption:** HbA1c 7%, no medication, so these terms add nothing |

These assumptions make this an approximate external validation. A UPCR of 0 cannot be
converted (log of 0), so those patients get a missing uACR.

In [ ]:
from egfr40 import ckd_epi_2021, pcr_to_acr

d = pd.DataFrame({"ID": route["ID"]})
d["age"] = route["age"]
d["sex"] = route["gender"].map({1: "male", 2: "female"})
d["sbp"] = route["SBP"]
d["bmi"] = route["BMI"]

# eGFR: CKD-EPI 2021 from serum creatinine (the model was built on it)
d["eGFR"] = ckd_epi_2021(route["Cr"], route["age"], (route["gender"] == 1).astype(int))

# uACR: UPCR g/gCr -> mg/g -> estimated uACR mg/g
d["uACR"] = pcr_to_acr(route["UPCR"] * 1000)

# Any antihypertensive class counts as BP medication
d["bp_meds"] = route[["use of RAASi", "use of CCB", "use of diuretics"]].max(axis=1)

# Predictors not recorded in CKD-ROUTE (see table above)
d["chd"] = route["prevalence of CVD"]
d["hf"] = 0
d["afib"] = 0
d["smoking"] = "never"

d["dm"] = route["diabetes"]
d["hba1c"] = np.where(d["dm"] == 1, 7.0, np.nan)
d["oral_dm"] = 0
d["insulin"] = 0

print(f"CKD-EPI 2021 eGFR median {d['eGFR'].median():.1f} vs JSN {route['eGFR'].median():.1f}")
d.isna().sum()[lambda s: s > 0]

## 3. The outcome: ≥40% eGFR decline or kidney failure within 3 years

The model predicts a composite of a ≥40% eGFR decline or kidney replacement therapy
(RRT) over 2 to 3 years. CKD-ROUTE measured eGFR every 6 months, so:

- **Event:** any visit up to 36 months with eGFR ≤ 60% of baseline, or RRT started
  within 36 months. The decline uses the study's own eGFR values, since it is relative
  to baseline and the equation cancels out.
- **Non-event:** no event and followed for at least 24 months. Patients who left
  earlier without an event have an unknown outcome and are excluded.

Patients with baseline eGFR < 15 are excluded, since they are already at kidney
failure levels and outside the model's population.

In [ ]:
visits = ["eGFR(6M)", "eGFR(12M)", "eGFR(18M)", "eGFR(24M)", "eGFR(30M)", "eGFR(36M)"]

# >=40% decline at any visit through 36 months (relative to baseline)
decline40 = route[visits].le(0.6 * route["eGFR(0M)"], axis=0).any(axis=1)
rrt36 = (route["RRT"] == 1) & (route["RRT_duration"] <= 36)
d["event"] = (decline40 | rrt36).astype(int)
followup = route["observational duration"]   # months

flow = [("All patients", len(d))]
keep = d["eGFR"] >= 15
flow.append(("Baseline eGFR >= 15", int(keep.sum())))
keep &= (d["event"] == 1) | (followup >= 24)
flow.append(("Known 3-year outcome (event, or followed >= 24 months)", int(keep.sum())))

cohort = d[keep].copy()
for step, n in flow:
    print(f"{step:<56} {n:>5,}")
print(f"\nEvents: {cohort['event'].sum():,} of {len(cohort):,} ({cohort['event'].mean():.1%})")

## 4. Score every patient

`add_egfr40_risk_col` gives each patient a 3-year risk from their own stratum.
Patients with a missing input (BMI, SBP or uACR) get `NaN` and are dropped.

In [ ]:
from egfr40 import add_egfr40_risk_col

cohort = add_egfr40_risk_col(cohort)
cohort["stratum"] = (
    cohort["dm"].map({0: "No DM", 1: "DM"})
    + ", eGFR "
    + cohort["eGFR"].ge(60).map({True: ">=60", False: "<60"})
)

scored = cohort.dropna(subset=["egfr40_3year"])
print(f"{len(scored):,} scored patients, {scored['event'].sum():,} events "
      f"({len(cohort) - len(scored):,} dropped for missing inputs)\n")
scored.groupby("stratum").agg(
    n=("event", "size"),
    events=("event", "sum"),
    observed=("event", "mean"),
    mean_predicted=("egfr40_3year", "mean"),
).round(3)

## 5. Discrimination and calibration with model_metrics

`model_metrics` takes the predicted probabilities (`y_prob`) and observed outcomes
(`y`) directly, so no fitted model is needed.

- **AUC-ROC:** how well the risk ranks patients who progressed above those who did not.
- **Precision-recall:** the same, focused on the progressors; the baseline is the
  event rate.
- **Calibration:** whether predicted risk matches the observed event rate. Expect some
  miscalibration: this is a referred CKD population, and several predictors are set
  to reference values.

In [ ]:
from model_metrics import (
    show_calibration_curve,
    show_pr_curve,
    show_roc_curve,
    summarize_model_performance,
)

y_prob = scored["egfr40_3year"].to_numpy()
y = scored["event"].to_numpy()

display(summarize_model_performance(y_prob=y_prob, y=y, model_title="egfr40",
                                    return_df=True))

show_roc_curve(y_prob=y_prob, y=y, model_title="egfr40", show_operating_point=True)
show_pr_curve(y_prob=y_prob, y=y, model_title="egfr40")
show_calibration_curve(y_prob=y_prob, y=y, model_title="egfr40", bins=10)